Is the energy elec calculation correct?

In [4]:
import logging
from pathlib import Path
from typing import Optional

import pytest
from numpy import isclose, number
from pyscf.lib.misc import StreamObject

import nbed
from nbed.driver import NbedDriver
from nbed.exceptions import NbedConfigError

water_xyz_raw = (
    "3\n \nH\t0.2774\t0.8929\t0.2544\nO\t0\t0\t0\nH\t0.6068\t-0.2383\t-0.7169"
)
args = {
    "geometry": water_xyz_raw,
    "n_active_atoms": 2,
    "basis": "6-31G*",
    "xc_functional": "b3lyp5",
    "projector": "mu",
    "localization": "ibo",
    "convergence": 1e-6,
    "run_ccsd_emb": True,
    "run_fci_emb": False,
}
driver = NbedDriver(**args, force_unrestricted=False)

 Iterative localization: IB/P4/2x2, 5 iter; Final gradient 1.04e-10
-4.338865034933823


In [5]:
print(driver.embedded_scf.mo_occ)

[2. 2. 2. 2. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [6]:
c_init = driver.embedded_scf.mo_coeff
c_init

array([[ 1.96451057e-03,  2.03424015e-01,  2.21519361e-01,
        -2.34073446e-06, -5.45135177e-02,  4.88387786e-02,
        -6.22580555e-01,  1.04671982e+00, -4.37437991e-05,
        -9.55787615e-02,  9.56202055e-02, -2.09266539e-01,
         7.35821218e-02,  1.71366759e-05, -4.80706860e-06,
        -7.79180849e-01,  9.73479515e-01],
       [ 5.36319071e-03,  4.49589186e-02,  1.29064312e-01,
        -2.66437879e-06, -9.48681956e-01,  1.40227529e+00,
         3.96466601e-01, -5.57551942e-01,  2.85059794e-05,
        -1.72806153e-01, -8.92067027e-01, -6.21999775e-01,
         4.26274058e-02,  7.64832414e-06, -4.89088478e-06,
        -1.15134104e-01, -4.05631270e-02],
       [ 1.00032181e+00, -1.82813338e-01,  7.44490214e-02,
        -1.15695421e-06, -8.52157851e-02,  2.36163262e-03,
         1.41603495e-02,  6.51622692e-02, -3.40327048e-06,
        -4.15889685e-02,  4.10630386e-03,  6.56323483e-02,
        -5.12114129e-04, -9.42986120e-07,  8.48589991e-07,
        -1.78487861e-02,  1.2

In [7]:
import numpy as np
from pyscf.lo import vvo
logger = logging.getLogger(__name__)

def virtual_valence(
    embedded_scf,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Localise virtual (unoccupied) orbitals using different localization schemes in PySCF.

    Args:
        embedded_scf. (StreamObject): PySCF method, with environment removed.

    Returns:
        embedded_scf. (StreamObject): Embedded PySCF method with localized virtual orbitals.
    """
    logger.debug("Beginning VVO Localisation.")
    if embedded_scf.mo_occ.shape[0] == 2:
        logger.debug("Unrestricted SCF, localising each spin independently.")
        embedded_scf.mo_coeff[0], embedded_scf.mo_occ[0] = _localize_virtual_spin(embedded_scf.mol, embedded_scf.mo_coeff[0], embedded_scf.mo_occ[0])
        embedded_scf.mo_coeff[1], embedded_scf.mo_occ[1] = _localize_virtual_spin(embedded_scf.mol, embedded_scf.mo_coeff[1], embedded_scf.mo_occ[1])
    else:
        embedded_scf.mo_coeff, embedded_scf.mo_occ = _localize_virtual_spin(embedded_scf.mol, embedded_scf.mo_coeff, embedded_scf.mo_occ)

def _localize_virtual_spin(mol, mo_coeff, mo_occ) -> np.ndarray:
    """Localise a single spin using VVO.
    
    Args:
        mol (Mole): PySCF molecule object.
        mo_coeff (np.ndarray): Molecular orbital coefficients.
        mo_occ (np.ndarray): Molecular orbital occupation numbers.

    Returns:
        mo_coeff (np.ndarray): Localised molecular orbital coefficients.
    """
    print(mo_coeff.shape)
    print(mo_occ.shape)
    c_std_occ = mo_coeff[:, mo_occ > 0]
    c_std_virt = mo_coeff[:, mo_occ == 0]

    c_virtual_loc = vvo.vvo(
        mol, c_std_occ, c_std_virt, iaos=None, s=None, verbose=None
    )
    print(f"{c_virtual_loc.shape[-1]} virtual valence orbitals.")

    logger.debug("Overwriting C Matrix and Occupancy.")
    mo_coeff = np.hstack((c_std_occ, c_virtual_loc))
    mo_occ = mo_occ[:mo_coeff.shape[-1]]

    return mo_coeff, mo_occ

In [11]:
virtual_valence(driver._mu["scf"])

(18, 7)
(7,)
3 virtual valence orbitals.


In [12]:
driver.embedded_scf.mo_occ


array([2., 2., 2., 2., 0., 0., 0.])

In [14]:
driver.embedded_scf.mo_coeff.shape

(18, 7)

In [15]:
from nbed.ham_builder import HamiltonianBuilder
import scipy as sp
from openfermion.linalg import get_sparse_operator

builder = HamiltonianBuilder(driver.embedded_scf, driver.classical_energy, "jordan_wigner")
tapered_ham = builder.build(taper=False)
loc_diag, _ = sp.sparse.linalg.eigsh(get_sparse_operator(tapered_ham), k=1, which="SA")


## Reference Values

In [16]:
driver._global_ccsd.e_tot

-76.20460941088893

In [17]:
driver._global_ks.e_tot

-76.36989408879548

## Unlocalised CCSD Embedding energy

In [18]:
driver._mu["e_ccsd"]

-76.20240053696659

In [19]:
loc_diag - driver._mu["e_ccsd"]

array([0.10500061])

In [20]:
loc_diag - driver._global_ccsd.e_tot

array([0.10720948])

Need some "External Molecular Orbitals"